# ANLP Assignment 3 - Part 1 & Part 2 Dual-GPU Kaggle Runner
This notebook automates repository setup, dataset linking from `/kaggle/input`, dual-GPU training & evaluation for Part 1, and Part 2 GNN experiments.

In [ ]:
import os
from pathlib import Path

# Clone repository if not already cloned
if not Path("anlp_assign_3").exists() and not Path("src").exists():
    !git clone https://github.com/AnishRacherla/anlp_assign_3.git
    %cd anlp_assign_3
elif Path("anlp_assign_3").exists() and not Path("src").exists():
    %cd anlp_assign_3

# Pull latest code and scripts
!git pull origin main

## 1. Setup Environment Variables (Credentials & Hyperparameters)
*Using `os.environ` in Python ensures variables persist across all cells in Kaggle.*

In [ ]:
import os

# Hugging Face & WandB Authentication
os.environ["HF_TOKEN"] = "hf_cFuvFxhXcCTIjYXZehTOKKsGNsNQPURzK"
os.environ["WANDB_API_KEY"] = "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"
os.environ["HF_REPO_ID"] = "AnishRacherla/anlp-a3-models"
os.environ["WANDB_PROJECT"] = "anlp-a3-part1"
os.environ["PYTHONUNBUFFERED"] = "1"

# Training Configuration
os.environ["EPOCHS"] = "3"
os.environ["BATCH_SIZE"] = "32"
os.environ["MAX_SEQ_LEN"] = "512"

# Optional: Set to run on a subset (e.g. 50000) for a rapid dry-run
# os.environ["MAX_TRAIN_SAMPLES"] = "50000"

print("Environment variables configured successfully!")

## 2. Locate & Link Dataset from `/kaggle/input`
*Recursively scans for `cipher.jsonl` anywhere inside `/kaggle/input` so nesting structure never causes issues.*

In [ ]:
from pathlib import Path
import shutil

print("Scanning /kaggle/input for cipher dataset...")
cipher_files = list(Path("/kaggle/input").rglob("cipher.jsonl"))

if not cipher_files:
    raise FileNotFoundError("Could not find any 'cipher.jsonl' in /kaggle/input! Please verify dataset is attached.")

print(f"Found {len(cipher_files)} cipher file(s):")
for cf in cipher_files:
    print(" -", cf)

Path("data/part1").mkdir(parents=True, exist_ok=True)

# Copy train, validation, and test splits
for split in ["train", "validation", "test"]:
    src_dir = None
    for cf in cipher_files:
        if split in cf.parts:
            src_dir = cf.parent
            break
            
    if src_dir and src_dir.exists():
        dst_dir = Path("data/part1") / split
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_dir, dst_dir)
        files = [p.name for p in dst_dir.glob("*")]
        print(f"Copied {split} -> {dst_dir} ({len(files)} files: {files})")
    else:
        print(f"Warning: Could not locate '{split}' split in /kaggle/input")

print("\nVerifying data/part1/train:")
!ls -lh data/part1/train/

## 3. Run Part 1: Dual-GPU Parallel Training & Evaluation
- **GPU 0**: Transformer $\rightarrow$ Hybrid (Mamba-DeltaNet)
- **GPU 1**: Mamba $\rightarrow$ DeltaNet

In [ ]:
!bash scripts/part1/run_kaggle.sh

## 4. Live Log Monitoring
*Run this cell anytime to see latest training progress on both GPUs.*

In [ ]:
print("=== GPU 0: Transformer Log ===")
!tail -n 15 logs/part1/gpu0_transformer.log 2>/dev/null || true

print("\n=== GPU 0: Hybrid Log ===")
!tail -n 15 logs/part1/gpu0_hybrid.log 2>/dev/null || true

print("\n=== GPU 1: Mamba Log ===")
!tail -n 15 logs/part1/gpu1_mamba.log 2>/dev/null || true

print("\n=== GPU 1: DeltaNet Log ===")
!tail -n 15 logs/part1/gpu1_deltanet.log 2>/dev/null || true

## 5. View Part 1 Metrics & Output Artifacts

In [ ]:
import json
from pathlib import Path

print("=== PART 1 EVALUATION RESULTS ===")
metrics_files = sorted(Path("results/part1").glob("*.json"))
if not metrics_files:
    print("No metric files found in results/part1 yet.")
for p in metrics_files:
    print(f"\n--- {p.name} ---")
    print(json.dumps(json.loads(p.read_text()), indent=2))

print("\n=== GENERATED PREDICTION ARTIFACTS ===")
!ls -lh output/part1/

## 6. Run Part 2: GNN Benchmark & Feature Ablations

In [ ]:
# 1. Run 5-seed model comparison & 4-feature GraphSAGE ablation
!bash scripts/part2/test_gnn.sh

# 2. Run TA split evaluation
!bash scripts/part2/test_gnn.sh --ta

## 7. View Part 2 Results

In [ ]:
import json
from pathlib import Path

for path_str in ["output/part2/results.json", "output/part2/results_ta.json", "results/part2/gnn_metrics_detailed.json"]:
    p = Path(path_str)
    if p.exists():
        print(f"\n=== {p} ===")
        print(json.dumps(json.loads(p.read_text()), indent=2))